In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
num_data_points = 500

# Mean and standard deviation for the normal distribution
mean_normal = [5, 10]
std_normal = [1, 2]

# Generate normal data points
normal_data_points = np.random.normal(loc=mean_normal, scale=std_normal, size=(num_data_points, 2))

# Introduce outliers by adding noise to the data
outliers = np.random.normal(loc=[20, 30], scale=[5, 8], size=(50, 2))

# Combine normal data points and outliers
data = np.vstack([normal_data_points, outliers])
target = np.hstack([np.zeros(num_data_points), np.ones(50)])

# Create a DataFrame for the dataset
df = pd.DataFrame(data, columns=['Feature1', 'Feature2'])
df['Outlier'] = target.astype(int)

# Visualize the dataset
plt.scatter(df['Feature1'], df['Feature2'], c=df['Outlier'], cmap='viridis')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Dummy Dataset for Outlier Detection')
plt.colorbar(label='Outlier (1: Yes, 0: No)')
plt.show()


In [ ]:
# Calculate Q1, Q3, and IQR for each feature
Q1 = df[['Feature1', 'Feature2']].quantile(0.25)
Q3 = df[['Feature1', 'Feature2']].quantile(0.75)
IQR = Q3 - Q1

# Define the outlier range based on IQR
outlier_range_lower = Q1 - 1.5 * IQR
outlier_range_upper = Q3 + 1.5 * IQR

# Identify outliers based on the IQR range
outliers_iqr = df[
    (df['Feature1'] < outlier_range_lower['Feature1']) |
    (df['Feature1'] > outlier_range_upper['Feature1']) |
    (df['Feature2'] < outlier_range_lower['Feature2']) |
    (df['Feature2'] > outlier_range_upper['Feature2'])
]

# Visualize the dataset with outliers detected by IQR method
plt.scatter(df['Feature1'], df['Feature2'], c='blue', label='Normal Data Points')
plt.scatter(outliers_iqr['Feature1'], outliers_iqr['Feature2'], c='red', label='Outliers (IQR)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Outlier Detection using IQR')
plt.legend()
plt.show()

In [ ]:
from sklearn.svm import OneClassSVM

# Combine the features into a single array
X = df[['Feature1', 'Feature2']].values

# Fit the One-Class SVM model
ocsvm = OneClassSVM(nu=0.15)  # Adjust the hyperparameter 'nu' based on your preference
ocsvm.fit(X)

# Predict outliers using the One-Class SVM model
outliers_ocsvm = ocsvm.predict(X)
outliers_ocsvm = outliers_ocsvm == -1  # Convert -1 to True (outlier), 1 to False (inlier)

# Visualize the dataset with outliers detected by One-Class SVM method
plt.scatter(X[:, 0], X[:, 1], c='blue', label='Normal Data Points')
plt.scatter(X[outliers_ocsvm, 0], X[outliers_ocsvm, 1], c='red', label='Outliers (One-Class SVM)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Outlier Detection using One-Class SVM')
plt.legend()
plt.show()


In [ ]:
from sklearn.ensemble import IsolationForest

# Combine the features into a single array
X = df[['Feature1', 'Feature2']].values

# Fit the Isolation Forest model
isolation_forest = IsolationForest(contamination=0.09)  # Adjust the contamination parameter based on your preference
isolation_forest.fit(X)

# Predict outliers using the Isolation Forest model
outliers_isolation_forest = isolation_forest.predict(X)
outliers_isolation_forest = outliers_isolation_forest == -1  # Convert -1 to True (outlier), 1 to False (inlier)

# Visualize the dataset with outliers detected by Isolation Forest method
plt.scatter(X[:, 0], X[:, 1], c='blue', label='Normal Data Points')
plt.scatter(X[outliers_isolation_forest, 0], X[outliers_isolation_forest, 1], c='red', label='Outliers (Isolation Forest)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Outlier Detection using Isolation Forest')
plt.legend()
plt.show()
